# 03b · 근거를 달고 답하는 질의응답 · 진단 · 다시 측정하기
**3세션 · 고르기 · RAG (오후)** — 실습 30분 + 고장 내기 20분 + 다시 측정하기

| 순서 | 할 일 |
|---|---|
| 1 청킹 · 2 색인 | 03a_rag_index 노트북에서 한 것 — 여기서 다시 만든다 |
| 3 ask() | 검색 → 주입 → 호출, 출처 각주 포함 |
| 4 거절 확인 | 답이 없는 질문 3개에 "자료에 없음"이 나오는가 — **건너뛰지 않는다** |
| 5 10문항 채점 | 답있음 5 · 답없음 3 · 바꿔쓴 2 |
| 6 고장 내기 | 짝끼리 — debug 출력만 보고 몇 번 단계 고장인지 말하기 |
| 8 창만 바꿔 다시 측정하기 | 자료 없음 · RAG · 자료 전부 |

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
print("모델:", llm.MODEL, "| 모의 모드" if llm.MOCK else "| 실제 호출")
from src import rag
print("임베딩:", "간이 해시 (EMBED=hash)" if rag.EMBED == "hash" else rag.ST_MODEL)
from src import qa

chunks = rag.chunk_folder("data/kb")
idx = rag.Index.build(chunks)
print("조각", len(chunks))

## 3. ask() — 검색 · 주입 · 호출
자료마다 번호와 출처를 달아 지시와 분리합니다. 규칙은 맨 앞, 질문은 맨 끝 — 어제 배운 위치.

In [ ]:
print(rag.RULES)
answer, hits = rag.ask("휴가 신청은 어떻게 해요?", idx, k=3)
print("\n" + answer)
print("\n각주:", rag.cited(answer), "→", [hits[n - 1].source for n in rag.cited(answer) if n <= len(hits)])

최종 프롬프트를 눈으로 봅니다. 모델이 본 것은 이것이 전부입니다.

In [ ]:
print(rag.LAST_PROMPT)

## 4. 거절 확인
지어낸 금액을 말하지 않는 것이 이 시스템의 **가장 중요한 동작**입니다.

In [ ]:
for q in ["올해 성과급 지급률은 몇 퍼센트인가요?", "사내 헬스장은 몇 시까지 운영하나요?", "육아휴직 기간에 급여는 얼마나 나오나요?"]:
    a, _ = rag.ask(q, idx)
    print(f"Q {q}\nA {a.strip()}\n")

> 세 번째 질문은 토론 거리입니다. 일반 업무 규칙 제18조는 "처우는 관계 법령에 따른다"고만 합니다. 이것은 *답*일까요, *자료에 없음*일까요? 테스트 문항으로 쓰려면 기준을 먼저 정해야 합니다.

## 5. 10문항 채점
어제와 같은 방식 — 고정 문항(`data/qa_tests.jsonl`) · 판정 기준 · 반복 기록.
다른 점은 실패를 **어느 단계의 고장인지**로 나눈다는 것입니다.
코드는 위치를 좁혀 줄 뿐입니다. *검색*인지 *청킹*인지는 `debug()`로 조각을 열어 사람이 가립니다 — 정답 문서가 아예 안 왔으면 검색, 왔는데 답이 잘려 있으면 청킹.

| 고장 | 판정 | 고치는 곳 |
|---|---|---|
| 검색 · 청킹 | 넣은 자료에 답이 든 조각이 없다 | 검색: k · 질문 말투 · 키워드 검색 / 청킹: 구조 경계 · 겹침 |
| 있는데 못 씀 | 자료에 답이 있는데 답에 없다 | 주입 코드 · 프롬프트 |
| 지어냄 | 답 없는 질문에 "자료에 없음"이 아니다 | 거절 지시 |
| 출처 오류 | 답은 맞는데 각주가 다른 문서를 가리킨다 | 각주 규칙 |

In [ ]:
passed, fails, rows = qa.run_qa(idx, k=3)

## 6. 진단 — 답만 보지 말고 중간을 찍는다

In [ ]:
FAIL_Q = rows[[r["ok"] for r in rows].index(False)]["q"] if fails else "휴가 쓰려면 며칠 전에 말해야 해요?"
rag.debug(FAIL_Q, idx, k=3);

## 7. 실습 · 일부러 고장 내기 (20분)
짝끼리 한 사람이 아래 고장 중 하나를 **몰래** 고르고, 다른 사람이 `debug` 출력만 보고 몇 번 단계 고장인지 맞힙니다.

| 고장 내는 법 | 기대 증상 | 진단한 단계 |
|---|---|---|
| ① `k=1`로 줄이고 단어를 바꾼 질문 | 답이 엉뚱함 | |
| ② `max_len=80`으로 잘게 자르기 | 문장이 끊김 | |
| ③ 자료를 프롬프트에서 빼먹는 버그 | 있는데 못 씀 | |
| ④ 규칙에서 "자료에 없음" 줄 지우기 | 지어냄 | |

In [ ]:
BREAK = 4      # ✏️ 1 · 2 · 3 · 4 중 하나 — 짝에게 보여 주지 않는다

broken_idx, kw = idx, {}
if BREAK == 1:
    kw = {"k": 1}
elif BREAK == 2:
    broken_idx = rag.Index.build(rag.chunk_folder("data/kb", max_len=80, overlap=0, pattern=None), name="broken")
elif BREAK == 3:
    kw = {"drop_context": True}
elif BREAK == 4:
    kw = {"rules": rag.RULES_NO_REFUSAL}

k = kw.pop("k", 3)
print("10문항 다시 채점")
_ = qa.run_qa(broken_idx, k=k, **kw)

In [ ]:
# 짝이 진단할 질문 하나 — debug 출력만 보고 몇 번 단계인지 말한다
rag.debug("사내 헬스장은 몇 시까지 운영하나요?" if BREAK == 4 else "휴가 쓰려면 며칠 전에 말해야 해요?",
          broken_idx, k=k, **kw);

**마지막 줄(④)을 꼭 해 봅니다.** 거절 지시 한 줄이 빠졌을 때 무엇이 나오는지 직접 보는 것이 가장 오래 남습니다.

고친 뒤에는 10문항을 다시 돌립니다 — 하나를 고치다 다른 하나가 깨지는 것은 RAG에서도 똑같이 일어납니다.

## 8. 창만 바꿔 다시 측정한다
같은 10문항, 같은 채점 기준으로 **창에 넣는 것만** 세 가지로 바꿉니다.

| 조건 | 창에 들어가는 것 |
|---|---|
| 창에 자료 없음 | 질문만 — 모델은 한빛정밀 규정을 모른다 |
| RAG | 검색한 조각 3개 + 거절 규칙 |
| 자료 전부 | 사내 자료 20편 전체 + 같은 규칙 (01 노트북의 A 조건) |

차이가 곧 **컨텍스트의 효과**이고, RAG와 *자료 전부*의 토큰 차이가 곧 **고르기의 값**입니다.

In [ ]:
ALL = "\n\n".join(f"<자료 출처=\"{name}\">\n{text}\n</자료>" for name, text in rag.load_folder("data/kb"))

def judge(answer, t):
    """답 문항은 정답 문자열, 답없음 문항은 "자료에 없음"이 있으면 통과 (출처는 보지 않는다)"""
    return qa.REFUSAL in answer if t.get("refuse") else any(e in answer for e in t["expect"])

def no_context(q):
    return llm.call(q, max_tokens=400)

def with_rag(q):
    hits = idx.search(q, 3)
    return llm.call(rag.make_prompt(q, hits), max_tokens=400)

def with_all(q):
    return llm.call(f"{rag.RULES}\n\n{ALL}\n\n<질문>{q}</질문>", max_tokens=400)

tests = qa.load_qa()
print(f"{'조건':<14}{'통과':>8}{'답없음 거절':>12}{'평균 입력 토큰':>16}{'평균 초':>9}  1건 비용")
for name, fn in [("창에 자료 없음", no_context), ("RAG", with_rag), ("자료 전부", with_all)]:
    rs = [(t, fn(t["q"])) for t in tests]
    ok = sum(judge(r.text, t) for t, r in rs)
    refused = sum(judge(r.text, t) for t, r in rs if t.get("refuse"))
    tin = sum(r.input_tokens for _, r in rs) / len(rs)
    sec = sum(r.seconds for _, r in rs) / len(rs)
    c = llm.cost(llm.MODEL, int(tin), int(sum(r.output_tokens for _, r in rs) / len(rs)))
    print(f"{name:<14}{ok:>5} / {len(rs)}{refused:>9} / 3{tin:>16,.0f}{sec:>9.1f}  {llm.fmt_cost(c)}")

| 조건 | 통과 | 답없음 거절 | 평균 입력 토큰 | 평균 초 |
|---|---|---|---|---|
| 창에 자료 없음 | | | | |
| RAG | | | | |
| 자료 전부 | | | | |

- *창에 자료 없음*의 답을 몇 개 열어 봅니다 — 모르면 그럴듯하게 채웁니다
- RAG가 *자료 전부*보다 낮은 문항이 있다면, 그 문항은 검색이 놓친 것입니다 → `rag.debug(...)`
- 하루 1,000건이면 RAG와 *자료 전부*의 비용 차이는 얼마인가